In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy.interpolate import griddata
import plotly.graph_objects as go


In [ ]:
# Load data
df = pd.read_csv("place: Synthetic_Gravity_Data : Here") #Download the data file and copy its path here
df

In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Scatter3d(
    x=df["longitude_deg"],
    y=df["latitude_deg"],
    z=df["bouguer_anomaly_mGal"],
    mode="markers",
    marker=dict(
        size=4,
        color=df["bouguer_anomaly_mGal"],  # color by anomaly
        colorscale="Viridis",
        colorbar=dict(title="Bouguer anomaly (mGal)")
    )
)])

fig.update_layout(
    scene=dict(
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        zaxis_title="Bouguer anomaly (mGal)"
    ),
    title="Raw Gravity Data (3D Scatter, No Interpolation)"
)

fig.show()

In [ ]:
lon = df["longitude_deg"].values
lat = df["latitude_deg"].values
bouguer = df["bouguer_anomaly_mGal"].values

# Create grid for surface (50x50 resolution)
lon_lin = np.linspace(lon.min(), lon.max(), 50)
lat_lin = np.linspace(lat.min(), lat.max(), 50)
LON_grid, LAT_grid = np.meshgrid(lon_lin, lat_lin)

# Interpolate onto grid
Bouguer_grid = griddata((lon, lat), bouguer, (LON_grid, LAT_grid), method="cubic")

# Create interactive 3D surface plot
fig = go.Figure(data=[go.Surface(
    z=Bouguer_grid,
    x=LON_grid,
    y=LAT_grid,
    colorscale="Viridis",
    colorbar=dict(title="Bouguer anomaly (mGal)")
)])

# Improve layout
fig.update_layout(
    scene=dict(
        xaxis_title="Longitude",
        yaxis_title="Latitude",
        zaxis_title="Bouguer anomaly (mGal)",
        aspectratio=dict(x=1, y=1, z=0.5)
    ),
    title="Interactive 3D Bouguer Anomaly Map"
)

fig.show()

In [ ]:
Bouguer_grid = np.nan_to_num(Bouguer_grid)

# 3. Apply 2D Fourier Transform
fft2d = np.fft.fft2(Bouguer_grid)
fft2d_shifted = np.fft.fftshift(fft2d)  # move zero frequency to center
amplitude_spectrum = np.abs(fft2d_shifted)

# 4. Plot amplitude spectrum
plt.figure(figsize=(8,6))
plt.imshow(np.log1p(amplitude_spectrum), cmap="inferno",
           extent=[-0.5, 0.5, -0.5, 0.5])  # normalized frequency axis
plt.colorbar(label="Log Amplitude")
plt.title("2D Fourier Spectrum of Bouguer Anomaly")
plt.xlabel("Frequency (lon direction)")
plt.ylabel("Frequency (lat direction)")
plt.show()


In [ ]:
fft2d = np.fft.fft2(Bouguer_grid)
fft2d_shifted = np.fft.fftshift(fft2d)
power_spectrum = np.abs(fft2d_shifted) ** 2

# 2. Frequency grid
n, m = Bouguer_grid.shape
u = np.arange(-n//2, n//2)
v = np.arange(-m//2, m//2)
U, V = np.meshgrid(u, v)
D = np.sqrt(U**2 + V**2).astype(int)  # radial distance (frequency bins)

# 3. Radial averaging
radial_power = np.bincount(D.ravel(), weights=power_spectrum.ravel())
radial_count = np.bincount(D.ravel())
radial_avg = radial_power / (radial_count + 1e-10)  # avoid divide by zero

# 4. Bar plot
plt.figure(figsize=(8,5))
plt.bar(np.arange(len(radial_avg)), radial_avg, width=1.0, color="steelblue")
plt.xlabel("Frequency (radial index)")
plt.ylabel("Average Power")
plt.title("Radial Power Spectrum of Gravity Data")
plt.show()